# 03 — Clean-Room SPAN 2x Super-Res & DeJPEG (~1.7 MB, Apache-2.0)
Trains pure-3x3-conv `CleanSpan2x` on **PD12M (CC0) + Infinite `e1_200` renders** with Charbonnier L1 + **LPIPS (BSD-2)** (zero DIV2K taint) and saves `span_2x_clean.onnx` straight to `MyDrive/infinite_ml/`.


In [ ]:
!pip -q uninstall -y torchaudio
!pip -q install -U torch torchvision onnx onnxscript onnxslim onnxruntime lpips datasets pillow numpy tqdm
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/out', exist_ok=True)
!unzip -q -o /content/drive/MyDrive/infinite_ml/e1_200.zip -d /content/


In [ ]:
import os, glob, random, shutil, hashlib, urllib.request
from io import BytesIO
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, onnx, lpips
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset

# Fetch 250 clean CC0 photos from Spawning/PD12M
pd_dir = '/content/pd12m_250'
os.makedirs(pd_dir, exist_ok=True)
if len(glob.glob(f'{pd_dir}/*.jpg')) < 200:
    ds_stream = load_dataset('Spawning/PD12M', split='train', streaming=True)
    saved = 0
    for row in ds_stream:
        if saved >= 250: break
        url = row.get('url') or row.get('image_url') or ''
        if not url.startswith('http'): continue
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'infinite-ml'})
            with urllib.request.urlopen(req, timeout=5) as r:
                im = Image.open(BytesIO(r.read())).convert('RGB')
            if min(im.size) < 256: continue
            im.resize((512, 512), Image.BILINEAR).save(f'{pd_dir}/pd_{saved:04d}.jpg', quality=92)
            saved += 1
        except Exception:
            continue
    print('Fetched PD12M CC0 photos:', saved)

class SPAB(nn.Module):
    def __init__(self, c=48):
        super().__init__()
        self.c1 = nn.Conv2d(c, c, 3, padding=1)
        self.c2 = nn.Conv2d(c, c, 3, padding=1)
        self.c3 = nn.Conv2d(c, c, 3, padding=1)
        self.act = nn.SiLU(inplace=True)
    def forward(self, x):
        h = self.act(self.c1(x))
        h = self.act(self.c2(h))
        h = self.c3(h)
        attn = torch.sigmoid(h) - 0.5
        return (x + h) * (1.0 + attn)

class CleanSpan2x(nn.Module):
    def __init__(self, c=48, blocks=6):
        super().__init__()
        self.head = nn.Conv2d(3, c, 3, padding=1)
        self.body = nn.ModuleList([SPAB(c) for _ in range(blocks)])
        self.fuse = nn.Conv2d(c, c, 3, padding=1)
        self.up = nn.Sequential(nn.Conv2d(c, 3 * 4, 3, padding=1), nn.PixelShuffle(2))
    def forward(self, x):
        base = F.interpolate(x, scale_factor=2.0, mode='bilinear', align_corners=False)
        h0 = self.head(x)
        h = h0
        for b in self.body: h = b(h)
        return torch.clamp(base + self.up(self.fuse(h) + h0), 0.0, 1.0)

hr_paths = sorted(glob.glob('/content/pd12m_250/*.jpg')) + sorted(glob.glob('/content/e1_200/scene_*/composite.png'))
print('Clean HR corpus:', len(hr_paths))

class SRDataset(Dataset):
    def __init__(self, paths, hr_sz=256):
        self.paths, self.hr_sz = paths, hr_sz
    def __len__(self): return len(self.paths) * 3
    def __getitem__(self, i):
        im = Image.open(self.paths[i % len(self.paths)]).convert('RGB')
        w, h = im.size
        x0, y0 = random.randint(0, max(0, w - self.hr_sz)), random.randint(0, max(0, h - self.hr_sz))
        hr = im.crop((x0, y0, x0 + self.hr_sz, y0 + self.hr_sz)).resize((self.hr_sz, self.hr_sz), Image.BICUBIC)
        lr = hr.resize((self.hr_sz // 2, self.hr_sz // 2), Image.BICUBIC)
        if random.random() < 0.6:
            buf = BytesIO(); lr.save(buf, format='JPEG', quality=random.randint(45, 92)); buf.seek(0)
            lr = Image.open(buf).convert('RGB')
        return (torch.from_numpy(np.asarray(lr, np.float32) / 255.0).permute(2,0,1),
                torch.from_numpy(np.asarray(hr, np.float32) / 255.0).permute(2,0,1))

dl_sr = DataLoader(SRDataset(hr_paths, 256), batch_size=16, shuffle=True, num_workers=2)
span = CleanSpan2x(c=48, blocks=6).cuda()
lpips_fn = lpips.LPIPS(net='alex').cuda()
for p in lpips_fn.parameters(): p.requires_grad = False
opt = torch.optim.AdamW(span.parameters(), lr=8e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=15)

for ep in range(15):
    tot_l1 = tot_psnr = 0.0
    for lr, hr in dl_sr:
        lr, hr = lr.cuda(), hr.cuda()
        pred = span(lr)
        l1 = torch.sqrt((pred - hr).pow(2) + 1e-6).mean()
        lp = lpips_fn(pred * 2 - 1, hr * 2 - 1).mean()
        loss = l1 + 0.03 * lp
        opt.zero_grad(); loss.backward(); opt.step()
        tot_l1 += l1.item()
        tot_psnr += -10.0 * np.log10(max(F.mse_loss(pred, hr).item(), 1e-8))
    sched.step()
    print(f"SPAN Epoch {ep+1}/15 l1={tot_l1/len(dl_sr):.4f} psnr={tot_psnr/len(dl_sr):.2f} dB")

out_span = '/content/out/span_2x_clean.onnx'
torch.onnx.export(span.eval().cpu(), torch.randn(1, 3, 256, 256), out_span, opset_version=17, dynamo=False,
                  input_names=['input'], output_names=['output'])
onnx.checker.check_model(out_span)
data = open(out_span, 'rb').read()
sha = hashlib.sha256(data).hexdigest()
shutil.copy2(out_span, '/content/drive/MyDrive/infinite_ml/span_2x_clean.onnx')
print(f"SPAN Exported {out_span}: {len(data)/1e6:.2f} MB sha256={sha}")
print("Saved to Drive: /content/drive/MyDrive/infinite_ml/span_2x_clean.onnx")
